# Этап M3 регрессия зарплаты по тексту (ruBERT)

Витрина: `data_for_models/df_m3_text.csv`.

Вход: `text_for_model`, таргет: `salary_from_log`.

Кросс-валидация: **GroupKFold** по `region_name`.

In [ ]:
import os
import random
import copy
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from pathlib import Path
from sklearn.metrics import mean_absolute_error

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import torch; print(f'CUDA доступна: {torch.cuda.is_available()}'); print(f'Versiya CUDA: {torch.version.cuda}')

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True  # фиксированная длина 512 — обычно выгодно
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("device:", device)

In [ ]:
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("AMP (mixed precision):", use_amp)

In [ ]:
df_m3_text = pd.read_csv("data_for_models/df_m3_text.csv", low_memory=False)
print(df_m3_text.shape)

In [ ]:
df_m3_text.head(5)

# Подбор параметров на подвыборке

In [ ]:
import itertools
import gc

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# --- Колонки и модель ---
MODEL_NAME = "DeepPavlov/rubert-base-cased"
TEXT_COL = "text_for_model"
TARGET_COL = "salary_from_log"
GROUP_COL = "region_name"

# Длина контекста: одинаково для уровня A и для финала (уровень C)
MAX_LENGTH = 512

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
MAX_GRAD_NORM = 1.0

# Стартовые значения (до подбора; подбор ниже перезапишет три строки)
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

# --- Подвыборка для уровня A ---
TUNE_FRAC = 0.2
TUNE_SEED = 42

TUNE_N_SPLITS = 2
TUNE_NUM_EPOCHS = 2

LR_GRID = (1e-5, 2e-5, 3e-5)
WD_GRID = (0.0, 0.01)
WU_GRID = (0.06, 0.1)

RUN_HPARAM_SEARCH = True  # после подбора поставьте False

In [ ]:
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class SalaryTextDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_length):
        self.texts = list(texts)
        self.targets = np.asarray(targets, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.targets[idx], dtype=torch.float32)
        return item


def collate_fn(batch):
    keys = batch[0].keys()
    out = {}
    for k in keys:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.cuda.amp import GradScaler, autocast


def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        problem_type="regression",
    )


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, y_true = [], []
    for batch in tqdm(loader, desc="valid", leave=False):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        labels = batch.pop("labels")
        with autocast(enabled=use_amp):
            out = model(**batch)
        logits = out.logits.squeeze(-1)
        preds.append(logits.float().cpu().numpy())
        y_true.append(labels.cpu().numpy())
    preds = np.concatenate(preds)
    y_true = np.concatenate(y_true)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mae = mean_absolute_error(y_true, preds)
    r2 = r2_score(y_true, preds)
    return rmse, mae, r2, preds, y_true


def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    n = len(loader.dataset)
    pbar = tqdm(loader, desc="train", leave=False)
    for batch in pbar:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            out = model(**batch)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * batch["input_ids"].size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / n


def state_dict_to_cpu(sd):
    return {k: v.detach().cpu().clone() for k, v in sd.items()}

In [ ]:
from sklearn.model_selection import GroupKFold

df_m3_text_full = df_m3_text.copy()

_tune_parts = []
for _, g in df_m3_text_full.groupby(GROUP_COL, sort=False):
    n_take = max(1, int(round(len(g) * TUNE_FRAC)))
    _tune_parts.append(g.sample(n=n_take, random_state=TUNE_SEED))

df_tune = (
    pd.concat(_tune_parts, axis=0)
    .sample(frac=1.0, random_state=TUNE_SEED)
    .reset_index(drop=True)
)

n_reg_full = df_m3_text_full[GROUP_COL].nunique()
n_reg_tune = df_tune[GROUP_COL].nunique()
assert n_reg_tune == n_reg_full, f"потеряны регионы: {n_reg_tune} vs {n_reg_full}"
assert TUNE_N_SPLITS <= n_reg_tune, "TUNE_N_SPLITS больше числа регионов в df_tune"

print("full rows:", len(df_m3_text_full), "| tune rows:", len(df_tune))
print("regions (full / tune):", n_reg_full, "/", n_reg_tune)

In [ ]:
if not RUN_HPARAM_SEARCH:
    print(
        "Пропуск подбора (RUN_HPARAM_SEARCH=False). Текущие LR / WD / WU:",
        LEARNING_RATE,
        WEIGHT_DECAY,
        WARMUP_RATIO,
    )
else:
    groups_t = df_tune[GROUP_COL].values
    y_t = df_tune[TARGET_COL].values
    texts_t = df_tune[TEXT_COL].astype(str).values

    gkf_tune = GroupKFold(n_splits=TUNE_N_SPLITS)
    pin_memory = device.type == "cuda"
    results = []

    for lr, wd, wu in itertools.product(LR_GRID, WD_GRID, WU_GRID):
        fold_best_val = []
        print(f"\n>>> cfg lr={lr:g} wd={wd} warmup={wu}")

        for fold, (train_idx, val_idx) in enumerate(gkf_tune.split(df_tune, y_t, groups_t)):
            train_ds = SalaryTextDataset(
                texts_t[train_idx], y_t[train_idx], tokenizer, MAX_LENGTH
            )
            val_ds = SalaryTextDataset(
                texts_t[val_idx], y_t[val_idx], tokenizer, MAX_LENGTH
            )
            train_loader = DataLoader(
                train_ds,
                batch_size=TRAIN_BATCH_SIZE,
                shuffle=True,
                collate_fn=collate_fn,
                num_workers=2,
                persistent_workers=True,
                pin_memory=pin_memory,
            )
            val_loader = DataLoader(
                val_ds,
                batch_size=EVAL_BATCH_SIZE,
                shuffle=False,
                collate_fn=collate_fn,
                num_workers=2,
                persistent_workers=True,
                pin_memory=pin_memory,
            )

            model = build_model().to(device)
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=lr, weight_decay=wd
            )
            scaler = GradScaler(enabled=use_amp)
            total_steps = len(train_loader) * TUNE_NUM_EPOCHS
            warmup_steps = int(total_steps * wu)
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps,
            )

            best_rmse = float("inf")
            best_state = None

            for epoch in range(TUNE_NUM_EPOCHS):
                train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
                rmse, mae, r2, _, _ = evaluate(model, val_loader)
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_state = state_dict_to_cpu(model.state_dict())
                print(
                    f"    fold {fold + 1}/{TUNE_N_SPLITS} ep {epoch + 1}/{TUNE_NUM_EPOCHS} | "
                    f"val_rmse(log)={rmse:.4f} | val_mae={mae:.4f} | R2={r2:.4f}"
                )

            if best_state is None:
                raise RuntimeError("нет best_state")

            model.load_state_dict(best_state)
            model.to(device)
            rmse_final, _, _, _, _ = evaluate(model, val_loader)
            fold_best_val.append(rmse_final)
            print(f"    fold {fold + 1} best_val_rmse={rmse_final:.4f}")

            del model, optimizer, scaler, scheduler, train_loader, val_loader
            del train_ds, val_ds, best_state
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()

        mean_rmse = float(np.mean(fold_best_val))
        results.append(
            {
                "lr": lr,
                "weight_decay": wd,
                "warmup_ratio": wu,
                "mean_val_rmse_log": mean_rmse,
                "fold_val_rmse_log": fold_best_val,
            }
        )
        print(f"=== mean val RMSE (log) = {mean_rmse:.4f}")

    best_row = min(results, key=lambda r: r["mean_val_rmse_log"])
    LEARNING_RATE = best_row["lr"]
    WEIGHT_DECAY = best_row["weight_decay"]
    WARMUP_RATIO = best_row["warmup_ratio"]

    print("\n--- Лучшая конфигурация (по среднему val RMSE на df_tune) ---")
    print(best_row)
    print("Подставлено в LEARNING_RATE, WEIGHT_DECAY, WARMUP_RATIO ")

# Основной прогон

In [ ]:
import os

# --- Финальный CV (уровень C) ---
N_SPLITS = 5
NUM_EPOCHS = 3
MAX_LENGTH = 512


print("Level C | N_SPLITS:", N_SPLITS, "| NUM_EPOCHS:", NUM_EPOCHS, "| MAX_LENGTH:", MAX_LENGTH)
print("Level C | LR:", LEARNING_RATE, "| WD:", WEIGHT_DECAY, "| warmup:", WARMUP_RATIO)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler
from sklearn.model_selection import GroupKFold
from transformers import get_linear_schedule_with_warmup

groups = df_m3_text[GROUP_COL].values
y = df_m3_text[TARGET_COL].values
texts = df_m3_text[TEXT_COL].astype(str).values

gkf = GroupKFold(n_splits=N_SPLITS)
fold_metrics = []
pin_memory = device.type == "cuda"

for fold, (train_idx, val_idx) in enumerate(gkf.split(df_m3_text, y, groups)):
    print(f"\n=== Fold {fold + 1}/{N_SPLITS} ===")

    train_ds = SalaryTextDataset(
        texts[train_idx], y[train_idx], tokenizer, MAX_LENGTH
    )
    val_ds = SalaryTextDataset(
        texts[val_idx], y[val_idx], tokenizer, MAX_LENGTH
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=pin_memory,
    )

    model = build_model().to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scaler = GradScaler(enabled=use_amp)

    total_steps = len(train_loader) * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_rmse = float("inf")
    best_state_cpu = None
    best_epoch = -1

    for epoch in range(NUM_EPOCHS):
        loss_avg = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
        rmse, mae, r2, _, _ = evaluate(model, val_loader)
        print(
            f"epoch {epoch + 1}/{NUM_EPOCHS} | train_loss={loss_avg:.4f} | "
            f"val_rmse(log)={rmse:.4f} | val_mae(log)={mae:.4f} | val_R2={r2:.4f}"
        )
        if rmse < best_rmse:
            best_rmse = rmse
            best_epoch = epoch + 1
            best_state_cpu = state_dict_to_cpu(model.state_dict())

    if best_state_cpu is None:
        raise RuntimeError("Не удалось сохранить лучший чекпоинт")

    model.load_state_dict(best_state_cpu)
    model.to(device)
    rmse_final, mae_final, r2_final, _, _ = evaluate(model, val_loader)

    fold_metrics.append(
        {
            "fold": fold + 1,
            "best_epoch": best_epoch,
            "val_rmse_log": rmse_final,
            "val_mae_log": mae_final,
            "val_r2": r2_final,
        }
    )

    os.makedirs("models/m3_rubert", exist_ok=True)
    torch.save(best_state_cpu, f"models/m3_rubert/fold_{fold + 1}.pt")

    del model, optimizer, scaler, scheduler, train_loader, val_loader, train_ds, val_ds, best_state_cpu
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\n=== Итог по фолдам ===")
display(pd.DataFrame(fold_metrics))

In [ ]:
fold_df = pd.DataFrame(
    {
        "fold": [1, 2, 3, 4, 5],
        "best_epoch": [3, 2, 3, 1, 3],
        "val_rmse_log": [0.262285, 0.231939, 0.254573, 0.227456, 0.242607],
        "val_mae_log": [0.195158, 0.167173, 0.190226, 0.160464, 0.176968],
        "val_r2": [0.707775, 0.688080, 0.680532, 0.698263, 0.683293],
    }
)
display(fold_df)

In [ ]:
print("\n=== CV summary (log-таргет) ===")
print(fold_df.to_string(index=False))
print("\nmean val_rmse_log:", fold_df["val_rmse_log"].mean())
print("std  val_rmse_log:", fold_df["val_rmse_log"].std(ddof=0))
print("mean val_mae_log:", fold_df["val_mae_log"].mean())
print("std  val_mae_log:", fold_df["val_mae_log"].std(ddof=0))
print("mean val_r2:     ", fold_df["val_r2"].mean())
print("std  val_r2:     ", fold_df["val_r2"].std(ddof=0))

In [ ]:
metrics_dir = Path("data_for_models")
metrics_dir.mkdir(parents=True, exist_ok=True)
results_row = {
    "model": "M3_text_rubert",
    "CV_RMSE_mean": float(fold_df["val_rmse_log"].mean()),
    "CV_RMSE_std": float(np.std(fold_df["val_rmse_log"].to_numpy(), ddof=0)),
    "CV_MAE_mean": float(fold_df["val_mae_log"].mean()),
    "CV_MAE_std": float(np.std(fold_df["val_mae_log"].to_numpy(), ddof=0)),
    "CV_R2_mean": float(fold_df["val_r2"].mean()),
    "CV_R2_std": float(np.std(fold_df["val_r2"].to_numpy(), ddof=0)),
    "n_folds": int(len(fold_df)),
}
results_df = pd.DataFrame([results_row])
m210_path = metrics_dir / "m2_10_cv_metrics.csv"
m210 = pd.read_csv(m210_path, encoding="utf-8-sig")
baseline_row = m210.loc[m210["Model"] == "M2.10.0_structure"].iloc[0]
baseline_rmse = float(baseline_row["CV_RMSE_mean"])
baseline_r2 = float(baseline_row["CV_R2_mean"])
results_df["RMSE_improvement_%"] = (
    baseline_rmse - results_df["CV_RMSE_mean"]
) / baseline_rmse * 100
results_df["R2_gain"] = results_df["CV_R2_mean"] - baseline_r2
preferred_cols = [
    "model",
    "CV_RMSE_mean",
    "CV_RMSE_std",
    "CV_MAE_mean",
    "CV_MAE_std",
    "CV_R2_mean",
    "CV_R2_std",
    "n_folds",
    "RMSE_improvement_%",
    "R2_gain",
]
m3_cv_metrics = results_df[[c for c in preferred_cols if c in results_df.columns]].copy()
m3_cv_metrics = m3_cv_metrics.rename(columns={"model": "Model"})
out_path = metrics_dir / "m3_text_cv_metrics.csv"
m3_cv_metrics.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved: {out_path}")
display(m3_cv_metrics.round(6))
fold_out = metrics_dir / "m3_text_cv_folds.csv"
fold_df.to_csv(fold_out, index=False, encoding="utf-8-sig")
print(f"Saved: {fold_out}")

# Сравним метрики моделей этапов М1, M2.1, M2.2, M2.3, M3

In [ ]:
metrics_dir = Path("data_for_models")
m1_cv_metrics = pd.read_csv(metrics_dir / "m1_cv_metrics.csv")
m2_cv_metrics = pd.read_csv(metrics_dir / "m2_cv_metrics.csv")
m25_cv_metrics = pd.read_csv(metrics_dir / "m2_5_cv_metrics.csv")
m210_cv_metrics = pd.read_csv(metrics_dir / "m2_10_cv_metrics.csv")
m3_cv_metrics = pd.read_csv(metrics_dir / "m3_text_cv_metrics.csv")
for df in (m1_cv_metrics, m2_cv_metrics, m25_cv_metrics, m210_cv_metrics, m3_cv_metrics):
    if "model" in df.columns:
        df.rename(columns={"model": "Model"}, inplace=True)
# --- M1
m1_part = m1_cv_metrics.loc[
    m1_cv_metrics["Model"].isin(
        ["OLS (sklearn pipeline, GroupKFold)", "Ridge (sklearn pipeline, GroupKFold)"]
    )
].copy()
m1_part["Stage"] = m1_part["Model"].map({
    "OLS (sklearn pipeline, GroupKFold)": "M1.1",
    "Ridge (sklearn pipeline, GroupKFold)": "M1.2",
})
m1_part["Algorithm"] = m1_part["Stage"].map({"M1.1": "OLS", "M1.2": "Ridge"})
m1_part["Features"] = "Structure"
# --- M2 (лучший spatial-конфиг)
m2_part = m2_cv_metrics.loc[m2_cv_metrics["Model"].isin(["M2.1.4_distance"])].copy()
m2_part["Stage"] = m2_part["Model"].map({"M2.1.4_distance": "M2.1.4"})
m2_part["Algorithm"] = "CatBoost"
m2_part["Features"] = "Structure + Spatial"
# --- M2.5
m25_part = m25_cv_metrics.loc[m25_cv_metrics["Model"].isin(["M2.5.2_full"])].copy()
m25_part["Stage"] = m25_part["Model"].map({"M2.5.2_full": "M2.5.2"})
m25_part["Algorithm"] = "CatBoost"
m25_part["Features"] = "Structure + Macro"
# --- M2.10
m210_part = m210_cv_metrics.loc[
    m210_cv_metrics["Model"].isin(["M2.10.2_geo_macro"])
].copy()
m210_part["Stage"] = m210_part["Model"].map({"M2.10.2_geo_macro": "M2.10.2"})
m210_part["Algorithm"] = "CatBoost"
m210_part["Features"] = "Structure + Spatial + Macro"
# --- M3 (текст / ruBERT) — имя Model как в m3_text_cv_metrics.csv
M3_MODEL_NAME = "M3_text_rubert"  # при необходимости замените на фактическое из CSV
m3_part = m3_cv_metrics.loc[m3_cv_metrics["Model"] == M3_MODEL_NAME].copy()
if m3_part.empty:
    raise ValueError(
        f"В m3_text_cv_metrics.csv нет строки Model == {M3_MODEL_NAME!r}. "
        "Проверьте имя модели в файле и поправьте M3_MODEL_NAME."
    )
m3_part["Stage"] = "M3"
m3_part["Algorithm"] = "ruBERT + head"
m3_part["Features"] = "Text"
common_cols = [
    "Model", "Stage", "Algorithm", "Features",
    "CV_RMSE_mean", "CV_RMSE_std", "CV_MAE_mean", "CV_MAE_std",
    "CV_R2_mean", "CV_R2_std", "n_folds",
]
m1_part = m1_part[[c for c in common_cols if c in m1_part.columns]].copy()
m2_part = m2_part[[c for c in common_cols if c in m2_part.columns]].copy()
m25_part = m25_part[[c for c in common_cols if c in m25_part.columns]].copy()
m210_part = m210_part[[c for c in common_cols if c in m210_part.columns]].copy()
m3_part = m3_part[[c for c in common_cols if c in m3_part.columns]].copy()
final_results_all = pd.concat(
    [m1_part, m2_part, m25_part, m210_part, m3_part],
    ignore_index=True,
)
stage_order = ["M1.1", "M1.2", "M2.1.4", "M2.5.2", "M2.10.2", "M3"]
final_results_all["Stage"] = pd.Categorical(
    final_results_all["Stage"], categories=stage_order, ordered=True
)
final_results_all = (
    final_results_all.sort_values("Stage").reset_index(drop=True)
)
display(final_results_all.round(6))

In [ ]:
out_path = metrics_dir / "cv_metrics_comparison_m1_m2_m25_m210_m3.csv"
final_results_all.to_csv(out_path, index=False)

In [ ]:
plot_df = final_results_all.copy()  # или final_results_all, если так назвали
sns.set(style="whitegrid", font_scale=1.1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(
    data=plot_df,
    x="Stage",
    y="CV_RMSE_mean",
    hue="Features",
    dodge=False,
    palette="Blues",
    ax=axes[0],
)
axes[0].set_title(
    "CV RMSE: M1 / M2 / M2.5 / M2.10 / M3 (согласованные конфигурации)"
)
axes[0].set_xlabel("Model Stage")
axes[0].set_ylabel("CV_RMSE_mean")
axes[0].legend(title="Features", loc="lower right")
axes[0].tick_params(axis="x", rotation=45)
for p in axes[0].patches:
    h = p.get_height()
    if pd.notna(h) and h > 0:
        axes[0].annotate(
            f"{h:.4f}",
            (p.get_x() + p.get_width() / 2, h),
            ha="center",
            va="bottom",
            fontsize=9,
            xytext=(0, 3),
            textcoords="offset points",
        )
sns.barplot(
    data=plot_df,
    x="Stage",
    y="CV_R2_mean",
    hue="Features",
    dodge=False,
    palette="Greens",
    ax=axes[1],
)
axes[1].set_title(
    "CV R²: M1 / M2 / M2.5 / M2.10 / M3 (согласованные конфигурации)"
)
axes[1].set_xlabel("Model Stage")
axes[1].set_ylabel("CV_R2_mean")
axes[1].legend(title="Features", loc="lower right")
axes[1].tick_params(axis="x", rotation=45)
for p in axes[1].patches:
    h = p.get_height()
    if pd.notna(h) and h > 0:
        axes[1].annotate(
            f"{h:.3f}",
            (p.get_x() + p.get_width() / 2, h),
            ha="center",
            va="bottom",
            fontsize=9,
            xytext=(0, 3),
            textcoords="offset points",
        )
plt.tight_layout()
plt.show()

In [ ]:
stage_order = ["M1.1", "M1.2", "M2.1.4", "M2.5.2", "M2.10.2", "M3"]
plot_df = plot_df.set_index("Stage").loc[stage_order].reset_index()
stages = [
    f"{row.Stage}\n{row.Algorithm}\n{row.Features.lower()}"
    for row in plot_df.itertuples(index=False)
]
rmse = plot_df["CV_RMSE_mean"].tolist()
r2 = plot_df["CV_R2_mean"].tolist()
x = np.arange(len(stages))
sns.set(style="whitegrid", context="talk")
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
axes[0].plot(x, r2, marker="o", linewidth=2.5)
axes[0].set_title("Сквозное сравнение: CV R² (6 стадий)")
axes[0].set_ylabel("CV R² mean")
axes[0].set_xticks(x)
axes[0].set_xticklabels(stages)
for i, v in enumerate(r2):
    axes[0].text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=14)
axes[1].plot(x, rmse, marker="o", linewidth=2.5, color="orange")
axes[1].set_title("Сквозное сравнение: CV RMSE (6 стадий)")
axes[1].set_xlabel("Model stage")
axes[1].set_ylabel("CV RMSE mean")
axes[1].set_xticks(x)
axes[1].set_xticklabels(stages)
for i, v in enumerate(rmse):
    axes[1].text(i, v + 0.001, f"{v:.3f}", ha="center", fontsize=14)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()